# Task 2 — OLTP or OLAP?

## 1. Classification
Test: **how many records does it touch?** and **right now, or over time?**

| # | Scenario | Type | Reason |
|---|---|---|---|
| 1 | Cashier scans chips, prints bill | OLTP | Touches one order and one stock row, needed right now. |
| 2 | CEO compares sales by category over 12 months | OLAP | Aggregates millions of rows across a year. |
| 3 | Customer updates phone number | OLTP | Updates one customer record immediately. |
| 4 | Eid week vs normal week, all branches | OLAP | Compares two periods over all branches; trend analysis. |
| 5 | Withdraw Rs 5,000 at ATM | OLTP | One account row updated; must be correct instantly. |
| 6 | Customers whose balance fell three months in a row | OLAP | Needs each customer's balance history, scanned across all customers. |
| 7 | Rider marks order delivered | OLTP | One status update on one order, now. |
| 8 | Head office ranks branches by returns this quarter | OLAP | Aggregates a quarter of returns across all branches. |

## 2. My own scenarios (Pakistani business: Easypaisa / Daraz)
- **OLTP — Easypaisa wallet transfer:** one sender and one receiver balance updated in a single short transaction, immediately, and it must never be half-done. Few rows, now.
- **OLAP — Daraz monthly sales by category and city:** reads every order line for months, groups and sums them; nobody is waiting on a single row, and the answer is about patterns over time.

## 3. Borderline: real-time card fraud check
- **For OLTP:** it runs on one card transaction and must answer in milliseconds, before the payment is approved.
- **For OLAP:** a decision on one transaction needs the card's history (typical amounts, cities, hours), which is an aggregation over time.
- **Decision:** put the *decision* on the OLTP side, but feed it **pre-computed history features** (e.g. 30-day average spend per card) calculated in the warehouse and refreshed periodically. Why: the live path must stay a few-row lookup; the heavy aggregation belongs on OLAP so it never delays payments.

## 4. Why the CEO's report slowed every billing counter
OLTP databases are optimised for **many tiny transactions**: normalised tables, indexes so each bill finds and changes a handful of rows, row-level locks, and caches tuned to keep the *hot* rows in memory.
The CEO's report is the opposite workload — a **full scan, join and aggregation over a year of rows**. On the same server it:
1. **Consumes disk I/O and CPU** for the scan, so billing reads/writes queue behind it.
2. **Flushes the buffer cache**: the scan pulls millions of cold pages in and evicts the hot pages billing needs, forcing billing to go to disk.
3. **Holds locks / long-running read snapshots** that make concurrent writes wait, or keep old row versions alive.
4. **Occupies connections and memory** (sorts, hash joins) taken from the pool the counters share.
Each bill is therefore slower, and since every counter is waiting on the same resources, every counter slows together. The database was designed to answer thousands of small questions per second, not one huge question.